In [ ]:
import argparse
import gzip
import pickle
import matplotlib.pyplot as plt
import yaml
import io
import os
import numpy as np
import hist
from typing import Any, IO, Dict, List, Iterable
import dctools
from dctools import plot as plotter
from dctools.plot import plotting
import matplotlib.pyplot as plt
import scipy.interpolate as interp
import mplhep as hep
from scipy import stats as st
np.seterr(all='warn')
plt.ioff()
from dctools import dict_to_hist_axis


In [ ]:
import sys, mplhep
print(sys.executable)
print(mplhep.__version__, mplhep.__file__)

In [ ]:
# %pip install --user "mplhep==1.3.0"

In [ ]:
config_2024 = dctools.read_config("config/inc-WZ/input_UL_2024-WZ_inclusive.yaml")
bh = config_2024.boosthist
print("datasets in pickle:", sorted(bh.keys()))

one = bh.get("WZto3LNu_TuneCP5_13p6TeV_powheg-pythia8")
# print(one)
if one is None:
    print("WZ sample not in pickle!")
else:
    print("observables:", sorted(one["hist"].keys())[:30])
    h = next(iter(one["hist"].values()))
    for ax in h.axes:
        print(ax.name, "->", list(ax)[:12] if ax.traits.discrete else (ax.edges[0], ax.edges[-1]))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import hist
import dctools
from dctools import plot as plotter
from dctools.plot import plotting

config_2024 = dctools.read_config("config/inc-WZ/input_UL_2024-WZ_inclusive.yaml")
print(config_2024.boosthist.keys())
# h = config_2018.boosthist["WZTo3LNu_TuneCP5_13TeV-amcatnloFXFX-pythia8"]["hist"]["dilep_tau_loose_met_hadron_mt"]
# name = "WZTo3LNu_TuneCP5_13TeV-amcatnloFXFX-pythia8"
# h['inc-SR0', 'nominal', :].project("dilep_tau_loose_met_hadron_mt").plot1d()

In [ ]:
print(one)

In [ ]:
AXIS_LABELS = {
    "dilep_tau_loose_met_hadron_mt": r"$M_{T}^{WZ}$ [GeV]",
    "mT_WZ":              r"$M_{T}^{WZ}$ [GeV]",
    "dilep_mt_llnunu":    r"$M_{T}^{\ell\ell\nu\nu}$ [GeV]",
    "dilep_m":            r"$m_{\ell\ell}$ [GeV]",
    "dilep_pt":           r"$p_{T}^{\ell\ell}$ [GeV]",
    "met_pt":             r"$p_{T}^{miss}$ [GeV]",
    "tau_pt_vtight":      r"$p_{T}^{\tau_h}$ [GeV]",
    "tau_pt_loose":       r"$p_{T}^{\tau_h}$ [GeV]",
    "qcos_theta_w_reco":  r"$q\cdot\cos\theta^{*}_{W}$",
    "cos_theta_z_reco":   r"$\cos\theta^{*}_{Z}$",
    "cos_theta_wp_reco":  r"$\cos\theta^{*}_{W^{+}}$",
    "cos_theta_wm_reco":  r"$\cos\theta^{*}_{W^{-}}$",
}

def plotting(config, variable, channel, rebin=1, xlim=[], blind=False, era="2024", checksyst=True,
             remap_replacement_types=None, logy=True, logx=False, bin_width_norm=None, no_ratios=False,
             combine_fit="pre-combine", combine_total_uncertainty="total_background",
             combine_channel_group=None, title="") -> None:
    assert combine_fit in ["pre-combine", "prefit", "fit_b", "fit_s"]
    if remap_replacement_types is None:
        remap_replacement_types = []                 # expected: "datadriven", "validation"
    if isinstance(remap_replacement_types, str):
        remap_replacement_types = [remap_replacement_types]
    datasets: Dict = dict()
    color_cycle: List = []

    if bin_width_norm is None and "bin_width_norm" in config:
        bin_width_norm = config.bin_width_norm

    for ng, name in enumerate(config.groups):
        histograms = dict(
            filter(
                lambda _n: _n[0] in config.groups[name].processes,
                config.boosthist.items()
            )
        )
        # keep per-helicity templates out of the data/MC stack
        if name.startswith(("WZ_W_", "WZ_Wp_", "WZ_Wm_", "WZ_Z_")):
            continue
        # skip groups with missing processes or without this observable
        if not histograms or not all(
            (variable in v["hist"]) if isinstance(v["hist"], dict)
            else (variable in [a.name for a in v["hist"].axes])
            for v in histograms.values()
        ):
            print(f"[skip] group {name}: missing processes or observable '{variable}'")
            continue

        p = dctools.datagroup(
            histograms       = histograms,
            ptype            = config.groups[name].type,
            observable       = variable,
            name             = name,
            xsections        = config.xsections,
            channel          = channel,
            luminosity       = config.luminosity.value,
            rebin            = rebin,
            remap_class_name = config.groups[name].remap_class_name if "remap_class_name" in config.groups[name] else None,
        )

        if p.remap_replace_group_name is not None:
            if p.remap_replace_type in remap_replacement_types:
                print(f"Overwriting: channel: {p.channel} type: {p.remap_replace_type}, {p.remap_replace_group_name} replaced by {p.name}")
                datasets[p.remap_replace_group_name] = p
                if hasattr(config.groups[name], "color") and len(p.to_boost().shape):
                    index = list(datasets.keys()).index(p.remap_replace_group_name)
                    color_cycle[index] = config.groups[name].color
            else:
                print(f"Skipping: channel: {p.channel} type: {p.remap_replace_type}, {p.remap_replace_group_name} would have been replaced by {p.name}")
                continue
        else:
            datasets[p.name] = p
            if hasattr(config.groups[name], "color") and len(p.to_boost().shape):
                color_cycle.append(config.groups[name].color)
        if p.ptype == "signal":
            signal = p.name

    if not datasets:
        raise RuntimeError(
            f"no groups survived for variable='{variable}', channel='{channel}' — "
            f"check pickle keys vs config processes, observable names, and channel names.")
    _plot_channel = plotter.add_process_axis(datasets)
    if variable not in _plot_channel.axes.name:
        raise ValueError(f"{variable} not found in histogram axes")

    variable_in_axes = variable
    pred = _plot_channel.project('process', 'systematic', variable_in_axes)[:hist.loc('data'), :, :]
    data = _plot_channel[{'systematic': 'nominal'}].project('process', variable_in_axes)[hist.loc('data'), :]
    proc_axis = pred.axes['process']

    for _proc in [str(_p) for _p in proc_axis]:
        print(f"{_proc} = ",
              _plot_channel[{'systematic': 'nominal'}].project('process', variable)[hist.loc(_proc), :].sum())

    fig, axes = plt.subplots(
        2, 1, figsize=(7, 7),
        gridspec_kw={"height_ratios": [3, 1]}, sharex=False)

    process_labels = {
        "WW": r"$WW \to \ell\nu\ell\nu$",
        "ZZ_ewk": r"$ZZ$ (EW production)",
        "WZ_ewk": r"$WZ$ (EW production)",
        "VVV": r"Triboson",
        "VBFZ": r"$Z$ (VBF)",
        "Top": r"$t\bar{t}$ + single $t$",
        "ZZ": r"$ZZ$",
        "DY": r"$Z/\gamma^*$",
        "WZ": r"$WZ$ (QCD production)",
    }
    # Petroff-10 color scheme
    cms_colors = {
        "DY":      "#ffa90e",
        "ZZ":      "#bd1f01",
        "WZ":      "#3f90da",
        "ZZ_ewk":  "#92dadd",
        "WZ_ewk":  "#b9ac70",
        "WW":      "#e76300",
        "Top":     "#94a4a2",
        "VVV":     "#a96b59",
        "VBFZ":    "#832db6",
    }

    # --- overlay: inclusive WZ (dotted), guarded ---
    if "WZ" in [str(p) for p in proc_axis]:
        signal_qcd = pred[{'systematic': 'nominal', 'process': 'WZ'}]
        unstacked_components  = [signal_qcd]
        unstacked_labels      = [r"$WZ$ signal"]
        unstacked_colors      = ["blue"]
        unstacked_kwargs_list = [{"linestyle": "dotted"}]
    else:
        unstacked_components = unstacked_labels = unstacked_colors = unstacked_kwargs_list = []

    hep.comp.data_model(
        data_hist=data.copy().reset() if blind else data,
        stacked_components=pred[{'systematic': 'nominal'}].stack('process'),
        stacked_labels=[process_labels.get(str(p), str(p)) for p in proc_axis],
        stacked_colors=[cms_colors.get(str(p), "#717581") for p in proc_axis],
        unstacked_components=unstacked_components,
        unstacked_labels=unstacked_labels,
        unstacked_colors=unstacked_colors,
        unstacked_kwargs_list=unstacked_kwargs_list,
        model_sum_kwargs={"show": False},
        xlabel="",
        ylabel="Events",
        comparison="split_ratio",
        data_label='Data',
        model_uncertainty_label="MC stat. unc.",
        fig=fig, ax_main=axes[0], ax_comparison=axes[1],
    )

    axes[0].set_yscale("log")
    axes[0].legend(ncol=2, fontsize=9, loc='upper right', frameon=False)
    # axes[0].text(0.03, 0.97, channel, transform=axes[0].transAxes,
    #              fontsize=11, va="top", ha="left", style="italic")
    axes[1].set_ylim(0, 2)
    axes[1].set_ylabel("Data / MC")
    axes[1].set_xlabel(AXIS_LABELS.get(variable, variable))
    if len(xlim) > 0:
        axes[0].set_xlim(xlim)
        axes[1].set_xlim(xlim)

    hep.cms.label(f"Preliminary — {c}", ax=axes[0], data=not blind,
                  lumi=config.luminosity.value, year=era, com=13.6)
    
    fig.savefig(f"{variable}_{channel}_{era}.pdf", bbox_inches="tight")

    return _plot_channel, datasets

In [ ]:
config_2024 = dctools.read_config("config/inc-WZ/input_UL_2024-WZ_inclusive.yaml")
y = "2024"
c = "inc-D0"
v = "dilep_pt"       
rrt = "validation" 
# rrt = ""
for channel in config_2024.plotting:
    ch_cfg = config_2024.plotting[channel]
    if (c not in channel): continue
    for vname in ch_cfg:
        if v not in vname: continue
        v_cfg = ch_cfg[vname]
        plotting(config_2024, 
                 vname, 
                 channel,
                 rebin=v_cfg.rebin, 
                 # rebin = 1,
                 xlim=v_cfg.range,
                 # xlim = (80,400),
                 blind=v_cfg.blind,
                 era="2024", 
                 remap_replacement_types=rrt
                )

In [ ]:
config_2024 = dctools.read_config("config/inc-WZ/input_UL_2024-WZ_inclusive.yaml")
y = "2024"
c = "inc-B01"
v = "dilep_tau_loose_met_hadron_mt"       
rrt = "datadriven" 
# rrt = ""
for channel in config_2024.plotting:
    ch_cfg = config_2024.plotting[channel]
    if (c not in channel): continue
    for vname in ch_cfg:
        if v not in vname: continue
        v_cfg = ch_cfg[vname]
        plotting(config_2024, vname, channel,
                 rebin=v_cfg.rebin, xlim=v_cfg.range, blind=v_cfg.blind,
                 era="2024", remap_replacement_types=rrt)

In [ ]:
config_2024 = dctools.read_config("config/inc-WZ/input_UL_2024-WZ_inclusive.yaml")
y = "2024"
c = "inc-D0"
v = "dilep_tau_loose_met_hadron_mt"       
# rrt = "validation" 

for channel in config_2024.plotting:
    ch_cfg = config_2024.plotting[channel]
    if (c not in channel): continue
    for vname in ch_cfg:
        if v not in vname: continue
        v_cfg = ch_cfg[vname]
        plotting(config_2024, vname, channel,
                 rebin=v_cfg.rebin, 
                 xlim=v_cfg.range,
                 # xlim = (0,400),
                 blind=v_cfg.blind,
                 era="2024", 
                 # remap_replacement_types=rrt
                )

In [ ]:
config_2024 = dctools.read_config("config/inc-WZ/input_UL_2024-WZ_inclusive.yaml")
y = "2024"
c = "inc-VR0"
v = "dilep_m"       
rrt = "datadriven" 

for channel in config_2024.plotting:
    ch_cfg = config_2024.plotting[channel]
    if (c not in channel): continue
    for vname in ch_cfg:
        if v not in vname: continue
        v_cfg = ch_cfg[vname]
        plotting(config_2024, vname, channel,
                 rebin=v_cfg.rebin, 
                 xlim=v_cfg.range,
                 # xlim = (0,400),
                 blind=v_cfg.blind,
                 era="2024", 
                 remap_replacement_types=rrt)

In [ ]:
import dctools
c = dctools.read_config("config/inc-WZ/input_UL_2024-WZ_inclusive.yaml")
print(c.boosthist.keys())

In [ ]:
proc = 'WZto3LNu_TuneCP5_13p6TeV_powheg-pythia8'

h2 = c.boosthist[proc]["hist"]["dilep_tau_loose_met_hadron_mt"]

print(h2)
